In [10]:
%matplotlib inline
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from importlib import reload

import os
import sys
# walk up from the cwd to the repo root, so re-running this cell after the chdir still works
PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "train_baseline.py").exists())
os.chdir(PROJECT_DIR)
for d in [PROJECT_DIR / "src", PROJECT_DIR / "src" / "splits"]:
    if str(d) not in sys.path:
        sys.path.insert(0, str(d))

import resplit_training as rt
import train_baseline as tb      # reuse the model, dataset and helpers from the training script

In [11]:
reload(tb)
# reload(rt)

<module 'train_baseline' from '/Users/miears.1/Library/CloudStorage/OneDrive-TheOhioStateUniversity/Erdos/autismCNN/autismCNN/src/train_baseline.py'>

In [12]:
MANIFEST = Path(f"{PROJECT_DIR}/data/manifest.csv")

manifest = pd.read_csv(MANIFEST)               
my_split = rt.make_split(manifest)
my_split.to_csv(f"{PROJECT_DIR}/data/manifest_split.csv", index=False)

In [13]:
print("Base split:")
print(    "Train:", len(my_split[my_split["split"] == "train"]),
          "Val:", len(my_split[my_split["split"] == "val"]),
          "Test:", len(my_split[my_split["split"] == "test"]))

Base split:
Train: 694 Val: 149 Test: 149


In [14]:
SPLITS = ["val"]                 # add "test" only for the final look

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
manifest = pd.read_csv(tb.MANIFEST_SPLIT)
print("device:", device)
orientations = ["axial", "sagittal", "coronal"]              
for orientation in orientations:
    print(f"Orientation: {orientation}")
    print(manifest[manifest[f"{orientation}_scan_id"].notnull()].groupby("split").agg(scans=(f"{orientation}_scan_id", "size"), subjects=("sub_id", "nunique")))

device: cpu
Orientation: axial
       scans  subjects
split                 
test      19        19
train     94        94
val       19        19
Orientation: sagittal
       scans  subjects
split                 
test     112       112
train    506       506
val      113       113
Orientation: coronal
       scans  subjects
split                 
test      33        33
train    159       159
val       34        34
